#  DurgotsavAI — Synthetic Data Generator

In [ ]:
# Step 1: Install and import libraries
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
print('✅ Libraries loaded!')

In [ ]:
# Step 2: Define pandals with base popularity scores
pandals = [
    {'name': 'Bagbazar Sarbojanin',       'lat': 22.5958, 'lon': 88.3697, 'base_popularity': 9},
    {'name': 'College Square',            'lat': 22.5790, 'lon': 88.3630, 'base_popularity': 9},
    {'name': 'Deshapriya Park',           'lat': 22.5263, 'lon': 88.3642, 'base_popularity': 8},
    {'name': 'Kumartuli Park',            'lat': 22.5950, 'lon': 88.3580, 'base_popularity': 7},
    {'name': 'Suruchi Sangha',            'lat': 22.5180, 'lon': 88.3470, 'base_popularity': 8},
    {'name': 'Sreebhumi Sporting Club',   'lat': 22.5780, 'lon': 88.4210, 'base_popularity': 9},
    {'name': 'Ekdalia Evergreen',         'lat': 22.5220, 'lon': 88.3560, 'base_popularity': 7},
    {'name': 'Tridhara Sammilani',        'lat': 22.5150, 'lon': 88.3620, 'base_popularity': 6},
    {'name': 'Badamtala Ashar Sangha',    'lat': 22.5350, 'lon': 88.3450, 'base_popularity': 7},
    {'name': 'Naktala Udayan Sangha',     'lat': 22.4890, 'lon': 88.3720, 'base_popularity': 6},
    {'name': 'Bosepukur Sitala Mandir',   'lat': 22.5050, 'lon': 88.3900, 'base_popularity': 6},
    {'name': 'Dum Dum Park Tarun Dal',    'lat': 22.6150, 'lon': 88.3980, 'base_popularity': 7},
    {'name': 'Santosh Mitra Square',      'lat': 22.5710, 'lon': 88.3560, 'base_popularity': 8},
    {'name': 'Mohammad Ali Park',         'lat': 22.5750, 'lon': 88.3510, 'base_popularity': 8},
    {'name': 'Jodhpur Park',              'lat': 22.5100, 'lon': 88.3690, 'base_popularity': 6},
]
print(f'✅ {len(pandals)} pandals defined!')

In [ ]:
# Step 3: Define Puja days and crowd pattern logic

# Puja days: Saptami to Dashami + Ekadashi (5 days)
puja_dates = [
    '2025-10-01',  # Saptami
    '2025-10-02',  # Ashtami
    '2025-10-03',  # Navami
    '2025-10-04',  # Dashami
    '2025-10-05',  # Ekadashi
]

# Day multipliers (Ashtami & Navami are busiest)
day_multiplier = {
    '2025-10-01': 0.7,
    '2025-10-02': 1.0,
    '2025-10-03': 1.0,
    '2025-10-04': 0.8,
    '2025-10-05': 0.5,
}

# Hourly crowd pattern (0-23 hours), peaks at evening
def hourly_factor(hour):
    if 0 <= hour < 6:    return 0.1   # late night / early morning
    elif 6 <= hour < 10: return 0.3   # morning
    elif 10 <= hour < 14: return 0.5  # afternoon
    elif 14 <= hour < 18: return 0.7  # evening build-up
    elif 18 <= hour < 22: return 1.0  # peak hours
    else:                 return 0.6  # late night

# Weather effect on crowd
weather_options = ['Clear', 'Cloudy', 'Light Rain', 'Heavy Rain']
weather_crowd_effect = {'Clear': 1.0, 'Cloudy': 0.9, 'Light Rain': 0.7, 'Heavy Rain': 0.4}

print('✅ Crowd pattern logic defined!')

In [ ]:
# Step 4: Generate the full dataset
np.random.seed(42)
records = []

MAX_CAPACITY = 10000  # max crowd per pandal per hour

for date_str in puja_dates:
    for hour in range(24):
        weather = np.random.choice(weather_options, p=[0.5, 0.25, 0.15, 0.10])
        w_effect = weather_crowd_effect[weather]
        d_mult = day_multiplier[date_str]
        h_factor = hourly_factor(hour)

        for pandal in pandals:
            base = pandal['base_popularity'] * 1000
            noise = np.random.normal(0, 300)
            crowd = int(base * h_factor * d_mult * w_effect + noise)
            crowd = max(0, min(crowd, MAX_CAPACITY))

            # Risk level based on crowd
            if crowd < 4000:
                risk = 'Green'
            elif crowd < 7000:
                risk = 'Yellow'
            else:
                risk = 'Red'

            records.append({
                'date': date_str,
                'hour': hour,
                'pandal_name': pandal['name'],
                'latitude': pandal['lat'],
                'longitude': pandal['lon'],
                'weather': weather,
                'base_popularity': pandal['base_popularity'],
                'crowd_count': crowd,
                'risk_level': risk,
            })

df = pd.DataFrame(records)
print(f'✅ Dataset generated: {len(df)} rows x {len(df.columns)} columns')
df.head(10)

In [ ]:
# Step 5: Quick summary stats
print('--- Risk Level Distribution ---')
print(df['risk_level'].value_counts())
print()
print('--- Average Crowd by Pandal ---')
print(df.groupby('pandal_name')['crowd_count'].mean().sort_values(ascending=False).round(0))

In [ ]:
# Step 6: Save to CSV
df.to_csv('durgotsavai_crowd_data.csv', index=False)
print('✅ Saved as durgotsavai_crowd_data.csv')
print('Download it from the Files panel on the left sidebar in Colab!')